
## STOCHASTIC MODELING
MODULE 7 | LESSON 2


---

# **TEMPORAL DIFFERENCE LEARNING**

|  |  |
|:---|:---|
|**Reading Time** |  60 min |
|**Prior Knowledge** |Markov process, Optimization, Monte Carlo  |
|**Keywords** |Reinforcement learning, multi-armed bandits


---

*In this last module, we are going to combine many of the concepts we have covered in the previous modules. We are going to delve further into three relevant RL algorithms, Monte Carlo, SARSA, and Q-learning, which exploit the dynamic programming principles and the simulation-based methods that we explored in previous modules. In this lesson, we focus on the SARSA and Q-learning methods.*

Before starting, let's reuse some code from the notebook of Lesson 1. 

In [1]:
import numpy as np
from numpy.random import rand, seed

In [2]:
N = 16
GRID_WIDTH = np.sqrt(N)
NROWS = int(N / GRID_WIDTH)
S_GRID = np.linspace(1, N - 2, N - 2)
A = 4

# Build an array that indicates, for each state, the destination cells
# from moving up, down, right, or left
destinations = np.zeros((N, A), dtype=np.int8)
destinations[N - 1, :] = (N - 1) * np.ones((A), dtype=np.int8)
for ss in range(1, N - 1):
    # determine row of position ss in the grid
    row_ss = np.floor(ss / GRID_WIDTH) + 1

    destinations[ss, 0] = (ss - GRID_WIDTH) * (ss - GRID_WIDTH >= 0) + ss * (
        ss - GRID_WIDTH < 0
    )
    destinations[ss, 1] = (ss + GRID_WIDTH) * (ss + GRID_WIDTH <= N - 1) + ss * (
        ss + GRID_WIDTH > N - 1
    )
    destinations[ss, 2] = (ss + 1) * (ss < row_ss * GRID_WIDTH - 1) + ss * (
        ss >= row_ss * GRID_WIDTH - 1
    )
    destinations[ss, 3] = (ss - 1) * (ss > (row_ss - 1) * GRID_WIDTH) + ss * (
        ss <= (row_ss - 1) * GRID_WIDTH
    )

print(destinations)

[[ 0  0  0  0]
 [ 1  5  2  0]
 [ 2  6  3  1]
 [ 3  7  3  2]
 [ 0  8  5  4]
 [ 1  9  6  4]
 [ 2 10  7  5]
 [ 3 11  7  6]
 [ 4 12  9  8]
 [ 5 13 10  8]
 [ 6 14 11  9]
 [ 7 15 11 10]
 [ 8 12 13 12]
 [ 9 13 14 12]
 [10 14 15 13]
 [15 15 15 15]]


In [3]:
# Set probability of down movement at each row

PDOWN = np.zeros((N))

PDOWN[0] = 0.8
PDOWN[1] = 0.4
PDOWN[2] = 0.2
PDOWN[3] = 0.1

In [4]:
def transition(state_init, action, dest, prdown, gridw, randnum):
    row_ss = np.floor(state_init / gridw)
    if randnum < prdown[int(row_ss)]:
        state_end = dest[int(state_init), 1]
    else:
        state_end = dest[int(state_init), action]
    return state_end

In [5]:
def exploring_starts(nstates, randnum):
    for ss in range(nstates):
        if randnum < (ss + 1) / nstates:
            break
    return ss


def e_greedy_policy(nactions, pol, eps, randnum1, randnum2, randnum3):
    if randnum1 < eps:
        for aa in range(nactions):
            if randnum2 < (aa + 1) / A:
                break
    else:
        for aa in range(nactions):
            if randnum3 < np.cumsum(pol)[aa]:
                break
    return aa

## **1. Temporal Difference Learning**

Temporal-Difference (TD) learning is a combination of Monte Carlo and dynamic programming (DP) notions of optimization. As in Monte Carlo methods, TD methods learn directly from experience without a model of the environment. As in DP, Temporal-Difference methods update estimates from states already updated, without waiting for a final outcome of an episode. This resembles the asynchronous dynamic programming technique that we described in Module 5 of this course. Some applications have very long episodes, so delaying all learning until the end of the episode slows down optimization. We consider two applications of Temporal Difference: SARSA and Q-Learning.

### **1.1 SARSA**

TD methods need to wait only until the next time step to update the increments to the optimization objects. At time $t$, they immediately make a useful update using the observed reward $r_t$ and the estimate of $Q$. The SARSA method starts by considering transitions from state-action pair to state-action pairs. That is, the estimation starts by a current observation of the state-action $(s_t,a_t)$, which yield a reward $r(s_t,a_t)$ and transition to state-action $(s_{t+1},a_{t+1})$. Importantly, the actions $a_t$ and $a_{t+1}$ arise from the optimal actions given states $s_t$ and $s_{t+1}$, respectively, using the current guess of policy, $\Pi$. Because of this, we refer to this algorithm as "on-policy" algorithm.

Given a vector of observations $(s_t,a_t,r_t,s_{t+1},a_{t+1})$, which gives the name to the optimization method, we can update the state-action value as:
$$
\begin{align*}
Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\big[r_t + \gamma Q(s_{t+1},a_{t+1}) - Q(s_t,a_t) \big]
\end{align*} 
$$
where the parameter $\alpha$ captures the speed of adjustment of our guesses to the arrival of new information, as in the bandit problems we studied in the previous module.


A pseudo-code for the SARSA algorithm is as follows:

0. Initialize arrays of policies, $\Pi$, and state-action values, $Q$.
1. Loop over each episode. Choose random initial state, $s_0$, and corresponding optimal action $a_0$, from $\varepsilon$-greedy policy. 

  * At each time step $t$, observe reward $r(s_t,a_t)$ and transition to following state $s_{t+1}$. 
  * Choose $a_{t+1}$ following $\varepsilon$-greedy policy. 
  * Update $Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\big[r_t + \gamma Q(s_{t+1},a_{t+1}) - Q(s_t,a_t) \big]$ and the policy $\Pi(s)=\arg\underset{a}{\max}Q(s,a)$.

To implement SARSA in the windy gridworld, we initialize the state-action value and policies. We also set the number of episodes, their maximum duration, the $\varepsilon$-greedy probability, and the updating parameter $\alpha$.

In [6]:
# Assume an initial policy that is random
policy = np.ones((N, A)) / A
qvalue = np.zeros((N, A))

# Array to fill with optimal action at each cell
policy0 = np.zeros((N))

EPISODES = 5000
EPISODE_TMAX = 100
EPSILON = 0.1
ALPHA = 0.1

In [7]:
# SARSA method in windy gridworld

seed(1234)

for episode in range(EPISODES):
    # Choose initial state s
    ss0 = exploring_starts(N, rand())
    # e-greedy policy to choose a
    aa = e_greedy_policy(A, policy[int(ss0), :], EPSILON, rand(), rand(), rand())
    # Use policy to generate history
    for tt in range(EPISODE_TMAX):
        qvalue_old = qvalue.copy()
        if ss0 < 1 or ss0 > N - 2:
            break
        # Determine s' given (s,a)
        ss1 = transition(ss0, aa, destinations, PDOWN, GRID_WIDTH, rand())
        # e-greedy policy to choose a' given s'
        aa_prime = e_greedy_policy(
            A, policy[int(ss0), :], EPSILON, rand(), rand(), rand()
        )
        if ss1 > 0 and ss1 < N - 1:
            qvalue[ss0, aa] = qvalue_old[ss0, aa] + ALPHA * (
                -1 + qvalue_old[ss1, aa_prime] - qvalue_old[ss0, aa]
            )
        else:
            qvalue[ss0, aa] = qvalue_old[ss0, aa] + ALPHA * (-1 - qvalue_old[ss0, aa])
        # (s',a') --> (s,a) in next time step
        ss0 = ss1
        aa = aa_prime
        # Improve policy
        policy0[ss0] = np.argmax(qvalue[ss0, :])
        policy[ss0, :] = np.zeros((A))
        policy[ss0, int(policy0[ss0])] = 1.0
        ss0 = ss1


# State-action values
print(qvalue)
# Optimal policy at each cell
for rr in range(NROWS):
    print(policy0[rr * int(GRID_WIDTH) : rr * int(GRID_WIDTH) + int(GRID_WIDTH)])

[[ 0.          0.          0.          0.        ]
 [-6.62508277 -7.60389126 -6.01375304 -6.19252633]
 [-5.35940971 -5.02716506 -4.86204842 -5.35661687]
 [-4.70036999 -3.2183337  -3.64994093 -4.42915708]
 [-3.20478699 -6.45921922 -7.05017357 -4.52288548]
 [-6.28045482 -5.42900443 -5.19242394 -6.83292171]
 [-5.66132332 -3.72901549 -3.62805378 -5.7825205 ]
 [-4.2907392  -2.0344024  -2.93365706 -4.10090066]
 [-5.62793024 -8.45092663 -6.76858456 -6.59375829]
 [-5.51362721 -4.17110515 -3.6046012  -6.80928323]
 [-4.47355548 -2.71821864 -2.75531089 -4.4470006 ]
 [-2.77205474 -1.         -1.80426489 -3.66254601]
 [-7.1151801  -4.5847541  -3.42928973 -4.69414687]
 [-5.09041322 -3.5056089  -2.20724889 -4.66837812]
 [-3.64162999 -2.16064915 -1.09027292 -3.72068896]
 [ 0.          0.          0.          0.        ]]
[0. 2. 2. 1.]
[0. 2. 2. 1.]
[0. 2. 1. 1.]
[2. 2. 2. 0.]


### **1.2 Q-learning**

Contrary to the "on-policy" SARSA algorithm introduced above, Q-learning is an "off-policy" method that is defined by the following updating criterion:
$$
\begin{align*}
Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\big[r_t + \gamma \underset{a}{\max}Q(s_{t+1},a) - Q(s_t,a_t) \big]
\end{align*} 
$$
That is, instead of choosing the action $a_{t+1}$ from the current policy (on-policy), we directly choose the action that maximizes the state-action value for the agent in step $t+1$ given the observed transition to state $s_{t+1}$ (off-policy). Notice that the current policy still determines which actions are followed and updated. This means that in step $t+1$ we observe $a_{t+1}$ as prescribed by the policy in state $s_{t+1}$, and not $\arg\underset{a}{\max}Q(s_{t+1},a)$.

A pseudo-code for the Q-learning algorithm is as follows:

0. Initialize arrays of policies, $\Pi$, and state-action values, $Q$.
1. Loop over each episode. Choose random initial state, $s_0$. 

  * At each time step $t$, choose optimal action $a_t$, from $\varepsilon$-greedy policy, observe reward $r(s_t,a_t)$ and transition to following state $s_{t+1}$. 
  * Update $Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\big[r_t + \gamma \underset{a}{\max}Q(s_{t+1},a) - Q(s_t,a_t) \big]$ and the policy $\Pi(s)=\arg\underset{a}{\max}Q(s,a)$.

We initialize the optimization objects and the parameters of the estimation as above.<span style='color: transparent; font-size:1%'>All rights reserved WQU WorldQuant University QQQQ</span>

In [8]:
# Assume an initial policy that is random
policy = np.ones((N, A)) / A
qvalue = np.zeros((N, A))

# Array to fill with optimal action at each cell
policy0 = np.zeros((N))

EPISODES = 5000
EPISODE_TMAX = 100
EPSILON = 0.1
ALPHA = 0.1

In [9]:
# Q-Learning method in windy gridworld

seed(1234)

for episode in range(EPISODES):
    # Choose initial state
    ss0 = exploring_starts(N, rand())
    # Use policy to generate history
    for tt in range(EPISODE_TMAX):
        qvalue_old = qvalue.copy()
        if ss0 < 1 or ss0 > N - 2:
            break
        # e-greedy policy to choose a
        aa = e_greedy_policy(A, policy[int(ss0), :], EPSILON, rand(), rand(), rand())
        # Determine s' given (s,a)
        ss1 = transition(ss0, aa, destinations, PDOWN, GRID_WIDTH, rand())
        if ss1 > 0 and ss1 < N - 1:
            qvalue[ss0, aa] = qvalue_old[ss0, aa] + ALPHA * (
                -1 + np.max(qvalue_old[ss1, :]) - qvalue_old[ss0, aa]
            )
        else:
            qvalue[ss0, aa] = qvalue_old[ss0, aa] + ALPHA * (-1 - qvalue_old[ss0, aa])
        # Improve policy
        policy0[ss0] = np.argmax(qvalue[ss0, :])
        policy[ss0, :] = np.zeros((A))
        policy[ss0, int(policy0[ss0])] = 1.0
        ss0 = ss1

# Optimal policy at each cell
for rr in range(NROWS):
    print(policy0[rr * int(GRID_WIDTH) : rr * int(GRID_WIDTH) + int(GRID_WIDTH)])

[0. 3. 2. 1.]
[0. 2. 2. 1.]
[2. 2. 2. 1.]
[2. 2. 2. 0.]



## **2. Conclusion**

In this lesson, we have worked through the concepts of temporal differential learning in reinforcement learning problems. In the next lesson, we will study the cliff walk with SARSA and q-learning methodologies.

See you there!

---
Copyright 2023 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
